In [2]:
# =================================================================================================
# NHANES 2015-2016 LABORATORY
# INSULIN - INS_I
#
# INPUT:
#   INS_I.xpt
#
# OUTPUT:
#   INS_I_corrected.csv
#
# COMPONENT:
#   Laboratory
#
# SUBCOMPONENT:
#   Insulin
#
#
# ATTACHED SOURCE FINDINGS
# -------------------------------------------------------------------------------------------------
#
# Rows                         = 3,191
# Columns                      = 7
#
# Variables:
#
#   SEQN
#   WTSAF2YR
#   LBXIN
#   LBDINSI
#   LBDINLC
#   PHAFSTHR
#   PHAFSTMN
#
# SEQN range                   = 83733-93697
# Missing SEQN                 = 0
# Duplicate SEQN               = 0
#
#
# VALIDATED SAS XPORT TINY-ZERO ARTIFACTS
# -------------------------------------------------------------------------------------------------
#
# WTSAF2YR       448
# LBDINLC      2,920
# PHAFSTHR        61
# PHAFSTMN        49
#
# TOTAL         3,478
#
#
# GENUINE DECIMAL LAB VARIABLES
# -------------------------------------------------------------------------------------------------
#
# WTSAF2YR
# LBXIN
# LBDINSI
#
#
# PROVEN INTEGRAL / CODE VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SEQN
# LBDINLC
# PHAFSTHR
# PHAFSTMN
#
#
# IMPORTANT WTSAF2YR RANGE NOTE
# -------------------------------------------------------------------------------------------------
#
# Actual attached XPT:
#
#   minimum positive = 13612.331812498
#   maximum positive = 521632.1858299
#
# CDC displays:
#
#   minimum positive = 13612.331812
#   maximum positive = 521632.18583
#
# The CDC values are displayed with lower decimal precision.
#
# We DO NOT change or round the actual XPT values.
#
# CDC range comparison therefore uses atol=1e-6.
#
# Exact numeric fidelity against the XPT is checked separately
# using zero-tolerance XPT -> CSV round-trip validation.
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# 1. Restore ONLY validated SAS XPORT tiny-zero artifacts.
#
# 2. Never globally replace tiny values.
#
# 3. Preserve genuine Lab decimal values.
#
# 4. Do NOT round.
#
# 5. Do NOT use float_format=.
#
# 6. Convert ONLY proven integral/code fields to nullable Int64.
#
# 7. Preserve missing-value positions exactly.
#
# 8. Validate CDC counts and displayed ranges.
#
# 9. Validate exact XPT -> CSV numeric round-trip fidelity.
#
# 10. Ensure SEQN has no ".0".
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


LAB_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)


XPT_FILE = (
    LAB_DIR
    / "INS_I.xpt"
)


CSV_FILE = (
    LAB_DIR
    / "INS_I_corrected.csv"
)


# =================================================================================================
# 3. VERIFY INPUT FILE EXISTS
# =================================================================================================

if not XPT_FILE.exists():

    raise FileNotFoundError(
        f"INS_I XPT file not found:\n{XPT_FILE}"
    )


# =================================================================================================
# 4. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched pandas-decoded source.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2015-2016 LABORATORY - INS_I"
)

print(
    "INSULIN XPT -> FIDELITY-PRESERVING CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Variables:"
)


print(
    df.columns.tolist()
)


# =================================================================================================
# 5. EXPECTED SOURCE STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI",

    "LBDINLC",

    "PHAFSTHR",

    "PHAFSTMN"

]


EXPECTED_ROWS = 3191


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "INS_I column structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected INS_I row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: INS_I structure = 3,191 rows x 7 columns."
)


# =================================================================================================
# 6. VERIFY ALL SOURCE FIELDS LOAD NUMERICALLY
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all seven INS_I variables "
        "to load numerically from XPT."
    )


print(
    "PASS: All seven source variables loaded numerically."
)


# =================================================================================================
# 7. SEQN SOURCE CHECK
# =================================================================================================

seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicate = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


seqn_min = int(
    df["SEQN"].min()
)


seqn_max = int(
    df["SEQN"].max()
)


print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicate
)


if seqn_min != 83733:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 93697:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicate != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 83733-93697 validated."
)


# =================================================================================================
# 8. DEFINE SAS XPORT TINY-ZERO VALUE
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


# =================================================================================================
# 9. FIND ALL TINY-XPORT VALUES
# =================================================================================================

tiny_counts = (

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    f"{tiny_total:,}"
)


# =================================================================================================
# 10. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

EXPECTED_TINY_COUNTS = {

    "WTSAF2YR": 448,

    "LBDINLC": 2920,

    "PHAFSTHR": 61,

    "PHAFSTMN": 49

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: INS_I tiny-value pattern differs "
        "from validated source.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3478:

    raise ValueError(
        "Unexpected total tiny-value count.\n"
        f"Expected: 3,478\n"
        f"Actual:   {tiny_total:,}"
    )


print(
    "PASS: Exact 3,478-value tiny-zero pattern validated."
)


# =================================================================================================
# 11. ORDINARY ZERO CHECK BEFORE REPAIR
# =================================================================================================

ordinary_zero_counts = (

    df[EXPECTED_COLUMNS]

    .eq(0)

    .sum()

)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORDINARY ZERO CHECK BEFORE REPAIR ---"
)


print(
    ordinary_zero_counts.to_string()
)


print(
    "\nOrdinary numeric zeros:",
    f"{ordinary_zero_total:,}"
)


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed in original INS_I XPT."
    )


print(
    "PASS: Validated zeros are represented by "
    "the SAS XPORT tiny value."
)


# =================================================================================================
# 12. BUILD EXPECTED SCIENTIFICALLY CORRECTED SOURCE
# =================================================================================================

#
# expected_df =
#
# original source
#
# +
#
# ONLY validated tiny-value -> zero restoration
#

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[

        expected_df[col]
        .eq(TINY_VALUE),

        col

    ] = 0


# =================================================================================================
# 13. RESTORE ONLY VALIDATED TRUE ZEROS
# =================================================================================================

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[
        col
    ] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "\nTotal restored zeros:",
    f"{total_corrections:,}"
)


if total_corrections != 3478:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "PASS: 3,478 validated zeros restored."
)


# =================================================================================================
# 14. VERIFY NO TINY VALUE REMAINS
# =================================================================================================

remaining_tiny = int(

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

    .sum()

)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT values remain."
)


# =================================================================================================
# 15. VERIFY CORRECTED ZERO PATTERN
# =================================================================================================

corrected_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if count > 0:

        corrected_zero_counts[
            col
        ] = count


print(
    "\n--- CORRECTED TRUE-ZERO DISTRIBUTION ---"
)


print(
    corrected_zero_counts
)


if corrected_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Corrected true-zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{corrected_zero_counts}"
    )


print(
    "PASS: Corrected zero distribution validated."
)


# =================================================================================================
# 16. CDC CHECK - WTSAF2YR COUNTS
# =================================================================================================

#
# CDC:
#
# Positive fasting weights = 2,743
# Zero weights             =   448
# Missing                  =     0
#

wtsaf = (
    df["WTSAF2YR"]
)


wtsaf_positive = int(

    (
        wtsaf > 0
    )

    .sum()

)


wtsaf_zero = int(

    wtsaf
    .eq(0)
    .sum()

)


wtsaf_missing = int(

    wtsaf
    .isna()
    .sum()

)


wtsaf_positive_values = (

    wtsaf.loc[
        wtsaf > 0
    ]

)


print(
    "\n--- CDC CHECK: WTSAF2YR ---"
)


print(
    "Positive:",
    f"{wtsaf_positive:,}"
)


print(
    "Zero:",
    f"{wtsaf_zero:,}"
)


print(
    "Missing:",
    f"{wtsaf_missing:,}"
)


if wtsaf_positive != 2743:

    raise ValueError(
        "WTSAF2YR positive count differs from CDC."
    )


if wtsaf_zero != 448:

    raise ValueError(
        "WTSAF2YR zero count differs from CDC."
    )


if wtsaf_missing != 0:

    raise ValueError(
        "WTSAF2YR missing count differs from CDC."
    )


print(
    "PASS: WTSAF2YR counts match CDC."
)


# =================================================================================================
# 17. WTSAF2YR CDC DISPLAYED RANGE CHECK
# =================================================================================================

#
# IMPORTANT:
#
# The actual XPT stores more precision than CDC displays.
#
# Actual attached XPT:
#
# minimum = 13612.331812498
# maximum = 521632.1858299
#
# CDC displayed:
#
# minimum = 13612.331812
# maximum = 521632.18583
#
# DO NOT round or modify the XPT values.
#
# This check only confirms agreement with CDC DISPLAY precision.
#

CDC_WTSAF2YR_MIN = 13612.331812

CDC_WTSAF2YR_MAX = 521632.18583


actual_wtsaf_min = float(
    wtsaf_positive_values.min()
)


actual_wtsaf_max = float(
    wtsaf_positive_values.max()
)


print(
    "\n--- WTSAF2YR RANGE CHECK ---"
)


print(
    "Actual XPT positive minimum:",
    repr(actual_wtsaf_min)
)


print(
    "CDC displayed minimum:",
    CDC_WTSAF2YR_MIN
)


print(
    "Difference:",
    abs(
        actual_wtsaf_min
        -
        CDC_WTSAF2YR_MIN
    )
)


print(
    ""
)


print(
    "Actual XPT positive maximum:",
    repr(actual_wtsaf_max)
)


print(
    "CDC displayed maximum:",
    CDC_WTSAF2YR_MAX
)


print(
    "Difference:",
    abs(
        actual_wtsaf_max
        -
        CDC_WTSAF2YR_MAX
    )
)


#
# CDC range values are display-rounded.
#
# 1e-6 tolerance affects ONLY codebook comparison.
#
# It does NOT affect source fidelity.
#

if not np.isclose(

    actual_wtsaf_min,

    CDC_WTSAF2YR_MIN,

    rtol=0,

    atol=1e-6

):

    raise ValueError(
        "WTSAF2YR positive minimum is outside "
        "CDC displayed-range tolerance."
    )


if not np.isclose(

    actual_wtsaf_max,

    CDC_WTSAF2YR_MAX,

    rtol=0,

    atol=1e-6

):

    raise ValueError(
        "WTSAF2YR positive maximum is outside "
        "CDC displayed-range tolerance."
    )


print(
    "PASS: WTSAF2YR range agrees with CDC "
    "after accounting for CDC display precision."
)


# =================================================================================================
# 18. CDC CHECK - LBXIN
# =================================================================================================

lbxin_nonmissing = int(

    df["LBXIN"]
    .notna()
    .sum()

)


lbxin_missing = int(

    df["LBXIN"]
    .isna()
    .sum()

)


lbxin_min = float(
    df["LBXIN"].min()
)


lbxin_max = float(
    df["LBXIN"].max()
)


print(
    "\n--- CDC CHECK: LBXIN ---"
)


print(
    "Nonmissing:",
    f"{lbxin_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbxin_missing:,}"
)


print(
    "Range:",
    lbxin_min,
    "to",
    lbxin_max
)


if lbxin_nonmissing != 2921:

    raise ValueError(
        "LBXIN nonmissing count differs from CDC."
    )


if lbxin_missing != 270:

    raise ValueError(
        "LBXIN missing count differs from CDC."
    )


if not np.isclose(

    lbxin_min,

    0.71,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBXIN minimum differs from CDC."
    )


if not np.isclose(

    lbxin_max,

    324.06,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBXIN maximum differs from CDC."
    )


print(
    "PASS: LBXIN matches CDC."
)


# =================================================================================================
# 19. CDC CHECK - LBDINSI
# =================================================================================================

lbdinsi_nonmissing = int(

    df["LBDINSI"]
    .notna()
    .sum()

)


lbdinsi_missing = int(

    df["LBDINSI"]
    .isna()
    .sum()

)


lbdinsi_min = float(
    df["LBDINSI"].min()
)


lbdinsi_max = float(
    df["LBDINSI"].max()
)


print(
    "\n--- CDC CHECK: LBDINSI ---"
)


print(
    "Nonmissing:",
    f"{lbdinsi_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbdinsi_missing:,}"
)


print(
    "Range:",
    lbdinsi_min,
    "to",
    lbdinsi_max
)


if lbdinsi_nonmissing != 2921:

    raise ValueError(
        "LBDINSI nonmissing count differs from CDC."
    )


if lbdinsi_missing != 270:

    raise ValueError(
        "LBDINSI missing count differs from CDC."
    )


if not np.isclose(

    lbdinsi_min,

    4.26,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBDINSI minimum differs from CDC."
    )


if not np.isclose(

    lbdinsi_max,

    1944.36,

    rtol=0,

    atol=1e-12

):

    raise ValueError(
        "LBDINSI maximum differs from CDC."
    )


print(
    "PASS: LBDINSI matches CDC."
)


# =================================================================================================
# 20. CDC CHECK - LBDINLC
# =================================================================================================

#
# CDC:
#
# 0 = At or above detection limit = 2,920
# 1 = Below detection limit       =     1
# Missing                         =   270
#

lbdinlc_counts = (

    df["LBDINLC"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


lbdinlc_missing = int(

    df["LBDINLC"]

    .isna()

    .sum()

)


EXPECTED_LBDINLC = {

    0.0: 2920,

    1.0: 1

}


print(
    "\n--- CDC CHECK: LBDINLC ---"
)


print(
    lbdinlc_counts
)


print(
    "Missing:",
    lbdinlc_missing
)


if lbdinlc_counts != EXPECTED_LBDINLC:

    raise ValueError(
        "LBDINLC distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_LBDINLC}\n\n"
        f"Actual:\n{lbdinlc_counts}"
    )


if lbdinlc_missing != 270:

    raise ValueError(
        "LBDINLC missing count differs from CDC."
    )


print(
    "PASS: LBDINLC matches CDC exactly."
)


# =================================================================================================
# 21. CDC CHECK - PHAFSTHR
# =================================================================================================

phafsthr_nonmissing = int(

    df["PHAFSTHR"]
    .notna()
    .sum()

)


phafsthr_missing = int(

    df["PHAFSTHR"]
    .isna()
    .sum()

)


phafsthr_zero = int(

    df["PHAFSTHR"]
    .eq(0)
    .sum()

)


phafsthr_min = float(
    df["PHAFSTHR"].min()
)


phafsthr_max = float(
    df["PHAFSTHR"].max()
)


print(
    "\n--- CDC CHECK: PHAFSTHR ---"
)


print(
    "Nonmissing:",
    f"{phafsthr_nonmissing:,}"
)


print(
    "Missing:",
    phafsthr_missing
)


print(
    "Zero:",
    phafsthr_zero
)


print(
    "Range:",
    phafsthr_min,
    "to",
    phafsthr_max
)


if phafsthr_nonmissing != 3135:

    raise ValueError(
        "PHAFSTHR nonmissing count differs from CDC."
    )


if phafsthr_missing != 56:

    raise ValueError(
        "PHAFSTHR missing count differs from CDC."
    )


if phafsthr_min != 0:

    raise ValueError(
        "PHAFSTHR minimum differs from CDC."
    )


if phafsthr_max != 37:

    raise ValueError(
        "PHAFSTHR maximum differs from CDC."
    )


if phafsthr_zero != 61:

    raise ValueError(
        "PHAFSTHR restored-zero count differs "
        "from validated source."
    )


print(
    "PASS: PHAFSTHR matches CDC/source."
)


# =================================================================================================
# 22. CDC CHECK - PHAFSTMN
# =================================================================================================

phafstmn_nonmissing = int(

    df["PHAFSTMN"]
    .notna()
    .sum()

)


phafstmn_missing = int(

    df["PHAFSTMN"]
    .isna()
    .sum()

)


phafstmn_zero = int(

    df["PHAFSTMN"]
    .eq(0)
    .sum()

)


phafstmn_min = float(
    df["PHAFSTMN"].min()
)


phafstmn_max = float(
    df["PHAFSTMN"].max()
)


print(
    "\n--- CDC CHECK: PHAFSTMN ---"
)


print(
    "Nonmissing:",
    f"{phafstmn_nonmissing:,}"
)


print(
    "Missing:",
    phafstmn_missing
)


print(
    "Zero:",
    phafstmn_zero
)


print(
    "Range:",
    phafstmn_min,
    "to",
    phafstmn_max
)


if phafstmn_nonmissing != 3135:

    raise ValueError(
        "PHAFSTMN nonmissing count differs from CDC."
    )


if phafstmn_missing != 56:

    raise ValueError(
        "PHAFSTMN missing count differs from CDC."
    )


if phafstmn_min != 0:

    raise ValueError(
        "PHAFSTMN minimum differs from CDC."
    )


if phafstmn_max != 59:

    raise ValueError(
        "PHAFSTMN maximum differs from CDC."
    )


if phafstmn_zero != 49:

    raise ValueError(
        "PHAFSTMN restored-zero count differs "
        "from validated source."
    )


print(
    "PASS: PHAFSTMN matches CDC/source."
)


# =================================================================================================
# 23. FRACTIONAL-VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() is NEVER applied back to the dataframe.
#

fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        fractional_count = 0


    else:

        fractional_count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


for col, count in fractional_counts.items():

    print(
        f"{col}: {count:,}"
    )


# =================================================================================================
# 24. FIELD TYPE PLAN
# =================================================================================================

DECIMAL_COLUMNS = [

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI"

]


INTEGER_COLUMNS = [

    "SEQN",

    "LBDINLC",

    "PHAFSTHR",

    "PHAFSTMN"

]


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Genuine Lab decimal fields:"
)


print(
    DECIMAL_COLUMNS
)


print(
    "Proven integral/code fields:"
)


print(
    INTEGER_COLUMNS
)


# =================================================================================================
# 25. VALIDATE DECIMAL CLASSIFICATION
# =================================================================================================

for col in DECIMAL_COLUMNS:

    if fractional_counts[col] == 0:

        raise ValueError(
            f"{col} unexpectedly contains no fractional values."
        )


print(
    "PASS: Genuine Lab decimal fields identified."
)


# =================================================================================================
# 26. VALIDATE INTEGER CLASSIFICATION
# =================================================================================================

for col in INTEGER_COLUMNS:

    if fractional_counts[col] != 0:

        raise ValueError(
            f"{col} unexpectedly contains fractional values."
        )


print(
    "PASS: Integral/code fields identified."
)


# =================================================================================================
# 27. VERIFY ONLY VALIDATED SOURCE VALUES CHANGED
# =================================================================================================

source_change_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_change_failures.append(
            col
        )


if source_change_failures:

    raise ValueError(
        "Unexpected source-value changes:\n"
        + str(
            source_change_failures
        )
    )


print(
    "PASS: Only validated tiny-zero values changed."
)


# =================================================================================================
# 28. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    corrected_missing = (

        df[col]

        .isna()

        .to_numpy()

    )


    missing_position_errors += int(

        (
            source_missing
            !=
            corrected_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed during correction."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 29. CONVERT ONLY PROVEN INTEGRAL/CODE FIELDS
# =================================================================================================

#
# This removes:
#
#   83733.0 -> 83733
#
# but NEVER converts:
#
#   WTSAF2YR
#   LBXIN
#   LBDINSI
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} contains unexpected fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Only proven integral/code fields converted to Int64."
)


# =================================================================================================
# 30. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 31. EXPORT CORRECTED LAB CSV
# =================================================================================================

#
# CRITICAL:
#
# NO:
#
#     float_format=
#
# NO:
#
#     round()
#
# NO integer conversion of genuine decimal Lab variables.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCorrected Laboratory CSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 32. RE-READ CSV USING ROUND-TRIP FLOAT PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 33. XPT -> CSV EXACT NUMERIC ROUND-TRIP FIDELITY
# =================================================================================================

#
# THIS is the strict source-fidelity check.
#
# No tolerance:
#
#     rtol = 0
#     atol = 0
#

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV exact numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All seven variables round-trip numerically exactly."
)


# =================================================================================================
# 34. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 35. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in SEQN/code fields."
)


# =================================================================================================
# 36. FINAL SEQN TEXT CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


final_seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if final_seqn_numeric.min() != 83733:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn_numeric.max() != 93697:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN clean."
)


# =================================================================================================
# 37. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 38. FINAL TRUE-ZERO DISTRIBUTION
# =================================================================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    zero_count = int(

        values
        .eq(0)
        .sum()

    )


    if zero_count > 0:

        final_zero_counts[
            col
        ] = zero_count


print(
    "\n--- FINAL TRUE-ZERO DISTRIBUTION ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "3,478 validated true zeros."
)


# =================================================================================================
# 39. SCIENTIFIC-NOTATION CHECK
# =================================================================================================

scientific_notation_fields = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


if scientific_notation_fields:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_fields
        )
    )


print(
    "PASS: No unexpected scientific notation."
)


# =================================================================================================
# 40. FINAL MISSING-POSITION FIDELITY
# =================================================================================================

final_missing_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    csv_missing = (

        raw_csv[col]

        .eq("")

        .to_numpy()

    )


    final_missing_errors += int(

        (
            source_missing
            !=
            csv_missing
        )

        .sum()

    )


if final_missing_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from source XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 41. FINAL DECIMAL PRESERVATION DIAGNOSTIC
# =================================================================================================

print(
    "\n--- LAB DECIMAL PRESERVATION ---"
)


for col in DECIMAL_COLUMNS:

    populated_count = int(

        raw_csv[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            raw_csv[col].ne("")
            &
            raw_csv[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: Genuine Lab decimal fields were not integerized."
)


# =================================================================================================
# 42. FINAL WTSAF2YR CHECK
# =================================================================================================

final_wtsaf = pd.to_numeric(

    raw_csv["WTSAF2YR"],

    errors="raise"

)


if int(
    (final_wtsaf > 0).sum()
) != 2743:

    raise ValueError(
        "Final WTSAF2YR positive count differs."
    )


if int(
    final_wtsaf.eq(0).sum()
) != 448:

    raise ValueError(
        "Final WTSAF2YR zero count differs."
    )


print(
    "PASS: Final WTSAF2YR:"
)


print(
    "  Positive = 2,743"
)


print(
    "  Zero     = 448"
)


# =================================================================================================
# 43. FINAL LBXIN CHECK
# =================================================================================================

if int(

    raw_csv["LBXIN"]

    .ne("")

    .sum()

) != 2921:

    raise ValueError(
        "Final LBXIN count differs from CDC."
    )


print(
    "PASS: Final LBXIN results = 2,921."
)


# =================================================================================================
# 44. FINAL LBDINSI CHECK
# =================================================================================================

if int(

    raw_csv["LBDINSI"]

    .ne("")

    .sum()

) != 2921:

    raise ValueError(
        "Final LBDINSI count differs from CDC."
    )


print(
    "PASS: Final LBDINSI results = 2,921."
)


# =================================================================================================
# 45. FINAL LBDINLC CHECK
# =================================================================================================

final_lbdinlc_counts = (

    raw_csv["LBDINLC"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


EXPECTED_FINAL_LBDINLC = {

    "0": 2920,

    "1": 1,

    "": 270

}


if final_lbdinlc_counts != EXPECTED_FINAL_LBDINLC:

    raise ValueError(
        "Final LBDINLC distribution differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_LBDINLC}\n\n"
        f"Actual:\n{final_lbdinlc_counts}"
    )


print(
    "PASS: Final LBDINLC:"
)


print(
    "  0       = 2,920"
)


print(
    "  1       = 1"
)


print(
    "  Missing = 270"
)


# =================================================================================================
# 46. FINAL FASTING TIME CHECK
# =================================================================================================

if int(

    raw_csv["PHAFSTHR"]

    .ne("")

    .sum()

) != 3135:

    raise ValueError(
        "Final PHAFSTHR populated count differs."
    )


if int(

    raw_csv["PHAFSTMN"]

    .ne("")

    .sum()

) != 3135:

    raise ValueError(
        "Final PHAFSTMN populated count differs."
    )


if int(

    raw_csv["PHAFSTHR"]

    .eq("0")

    .sum()

) != 61:

    raise ValueError(
        "Final PHAFSTHR zero count differs."
    )


if int(

    raw_csv["PHAFSTMN"]

    .eq("0")

    .sum()

) != 49:

    raise ValueError(
        "Final PHAFSTMN zero count differs."
    )


print(
    "PASS: Final fasting-time fields validated."
)


# =================================================================================================
# 47. FINAL NUMERIC RANGE RECHECK
# =================================================================================================

#
# Strictly compare the CSV back to the corrected XPT values.
#
# This proves we did NOT replace:
#
#     13612.331812498
#
# with:
#
#     13612.331812
#

final_positive_wtsaf = (

    final_wtsaf.loc[
        final_wtsaf > 0
    ]

)


final_min = float(
    final_positive_wtsaf.min()
)


final_max = float(
    final_positive_wtsaf.max()
)


print(
    "\n--- FINAL WTSAF2YR SOURCE-PRECISION CHECK ---"
)


print(
    "Final CSV minimum:",
    repr(final_min)
)


print(
    "Corrected XPT minimum:",
    repr(actual_wtsaf_min)
)


print(
    "Final CSV maximum:",
    repr(final_max)
)


print(
    "Corrected XPT maximum:",
    repr(actual_wtsaf_max)
)


if final_min != actual_wtsaf_min:

    raise ValueError(
        "WTSAF2YR minimum changed during CSV export."
    )


if final_max != actual_wtsaf_max:

    raise ValueError(
        "WTSAF2YR maximum changed during CSV export."
    )


print(
    "PASS: WTSAF2YR source precision preserved."
)


# =================================================================================================
# 48. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2015-2016 LABORATORY INS_I CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Component:"
)


print(
    "  Laboratory"
)


print(
    ""
)


print(
    "Lab file:"
)


print(
    "  INS_I - Insulin"
)


print(
    ""
)


print(
    "Cycle:"
)


print(
    "  2015-2016"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(raw_csv):,}"
)


print(
    "Columns:",
    len(raw_csv.columns)
)


print(
    "Unique SEQN:",
    f"{raw_csv['SEQN'].nunique():,}"
)


print(
    "SEQN range:"
)


print(
    "  83733-93697"
)


print(
    ""
)


print(
    "Validated true-zero restorations:"
)


print(
    "  WTSAF2YR =   448"
)


print(
    "  LBDINLC  = 2,920"
)


print(
    "  PHAFSTHR =    61"
)


print(
    "  PHAFSTMN =    49"
)


print(
    "  ----------------"
)


print(
    "  TOTAL     = 3,478"
)


print(
    ""
)


print(
    "Genuine Lab decimal fields preserved:"
)


print(
    "  WTSAF2YR"
)


print(
    "  LBXIN"
)


print(
    "  LBDINSI"
)


print(
    ""
)


print(
    "Integral/code fields cleaned:"
)


print(
    "  SEQN"
)


print(
    "  LBDINLC"
)


print(
    "  PHAFSTHR"
)


print(
    "  PHAFSTMN"
)


print(
    ""
)


print(
    "WTSAF2YR:"
)


print(
    "  Positive = 2,743"
)


print(
    "  Zero     = 448"
)


print(
    "  XPT positive minimum:",
    repr(actual_wtsaf_min)
)


print(
    "  XPT positive maximum:",
    repr(actual_wtsaf_max)
)


print(
    "  CDC displayed minimum:",
    CDC_WTSAF2YR_MIN
)


print(
    "  CDC displayed maximum:",
    CDC_WTSAF2YR_MAX
)


print(
    ""
)


print(
    "LBXIN results:"
)


print(
    "  2,921"
)


print(
    ""
)


print(
    "LBDINSI results:"
)


print(
    "  2,921"
)


print(
    ""
)


print(
    "LBDINLC:"
)


print(
    "  0 = 2,920"
)


print(
    "  1 = 1"
)


print(
    "  Missing = 270"
)


print(
    ""
)


print(
    "PHAFSTHR populated:"
)


print(
    "  3,135"
)


print(
    ""
)


print(
    "PHAFSTMN populated:"
)


print(
    "  3,135"
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Remaining tiny XPORT artifacts: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Unexpected source-value changes: 0"
)


print(
    "Lab decimal integerization: 0"
)


print(
    "WTSAF2YR CDC display-range check: PASS"
)


print(
    "XPT -> CSV exact numeric round-trip: PASS"
)


print(
    "=" * 110
)

NHANES 2015-2016 LABORATORY - INS_I
INSULIN XPT -> FIDELITY-PRESERVING CSV
Rows: 3,191
Columns: 7
Variables:
['SEQN', 'WTSAF2YR', 'LBXIN', 'LBDINSI', 'LBDINLC', 'PHAFSTHR', 'PHAFSTMN']
PASS: INS_I structure = 3,191 rows x 7 columns.
PASS: All seven source variables loaded numerically.

--- SEQN SOURCE CHECK ---
Minimum: 83733
Maximum: 93697
Missing: 0
Duplicates: 0
PASS: SEQN range 83733-93697 validated.

--- ORIGINAL XPORT TINY-ZERO CHECK ---
SEQN           0
WTSAF2YR     448
LBXIN          0
LBDINSI        0
LBDINLC     2920
PHAFSTHR      61
PHAFSTMN      49

Total tiny XPORT values: 3,478
PASS: Exact 3,478-value tiny-zero pattern validated.

--- ORDINARY ZERO CHECK BEFORE REPAIR ---
SEQN        0
WTSAF2YR    0
LBXIN       0
LBDINSI     0
LBDINLC     0
PHAFSTHR    0
PHAFSTMN    0

Ordinary numeric zeros: 0
PASS: Validated zeros are represented by the SAS XPORT tiny value.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR: 448
LBDINLC: 2,920
PHAFSTHR: 61
PHAFSTMN: 49

Total restored zeros: